In [5]:
import torch, psutil, os

print(f"CPU RAM: {psutil.virtual_memory().available / 1e9:.1f} GB 남음")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB 전체")
    print(f"GPU VRAM: {torch.cuda.memory_reserved(0) / 1e9:.1f} GB 사용 중")
else:
    print("GPU 없음 — CPU 전용 실행")

# ─── Cell 2: 설정값 정의 ─────────────────────────────────────────────────────
import torch
model_name   = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file   = r'C:\Users\user\Desktop\hun\LLM_study\finetune_data\train.jsonl'
valid_file   = r'C:\Users\user\Desktop\hun\LLM_study\finetune_data\validation.jsonl'
output_dir   = 'peft_output'
num_train_epochs           = 3
per_device_train_batch_size = 4
learning_rate              = 2e-4
use_bf16 = True if torch.cuda.is_available() else False
use_fp16 = not use_bf16
lora_r       = 8
lora_alpha   = 16
lora_dropout = 0.05
use_4bit     = False
max_leng     = 512

LABEL_DESC = {
    'DEF':  '정의/목적/적용범위 조항',
    'RIGHT':'권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG':  '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC':  '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

# ─── Cell 3: 데이터 전처리 함수 정의 ─────────────────────────────────────────
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt 부분은 loss에서 제외(-100), 정답 라벨 토큰만 학습'''
    inputs, labels = [], []
    for p, r in zip(batch['prompt'], batch['response']):
        full            = p + ' ' + r
        tokenized_full   = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p,    truncation=True, max_length=max_length)
        input_ids = tokenized_full['input_ids']
        label_ids = input_ids.copy()
        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i] = -100       # prompt 마스킹
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids': inputs, 'labels': labels}

# ─── Cell 4: 토크나이저 & 모델 로드 ──────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code': True}
if use_4bit:
    load_kwargs.update({'load_in_4bit': True, 'device_map': 'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

# ─── Cell 5: LoRA 타겟 모듈 자동 탐색 & PEFT 설정 ────────────────────────────
import torch.nn as nn

def find_lora_target_modules(model):
    linear_leaf_names = {name.split('.')[-1]
                         for name, module in model.named_modules()
                         if isinstance(module, nn.Linear)}
    preferred = ['q_proj','k_proj','v_proj','o_proj',
                 'gate_proj','up_proj','down_proj']
    selected = [m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([n for n in linear_leaf_names
                           if any(k in n for k in fallback)
                           and n not in {'lm_head'}])
    if not selected:
        raise ValueError(f'LoRA 타겟 모듈 자동 탐색 실패: {linear_leaf_names}')
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Selected target_modules for LoRA:', target_modules)

peft_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ─── Cell 6: 데이터셋 토크나이징 & 패딩 ─────────────────────────────────────
data_files   = {'train': train_file, 'validation': valid_file}
dataset      = load_dataset('json', data_files=data_files)
preprocess   = lambda x: build_inputs_and_labels(x, tokenizer, max_length=max_leng)
tokenized_train = dataset['train'].map(
    preprocess, batched=True, remove_columns=dataset['train'].column_names)
tokenized_eval  = dataset['validation'].map(
    preprocess, batched=True, remove_columns=dataset['validation'].column_names)

def collate_with_padding(features):
    input_ids = [f['input_ids'] for f in features]
    labels    = [f['labels']    for f in features]
    padded    = tokenizer.pad({'input_ids': input_ids},
                              padding=True, return_tensors='pt')
    max_len   = padded['input_ids'].shape[1]
    padded_labels = [lb + [-100] * (max_len - len(lb)) for lb in labels]
    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded

print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

# ─── Cell 7: 학습 실행 ────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
base_kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    bf16=use_bf16, fp16=use_fp16,
    save_total_limit=3,
    remove_unused_columns=False,
    logging_steps=10,
)
try:
    training_args = TrainingArguments(**{**base_kwargs, 'evaluation_strategy': 'epoch'})
except TypeError:
    training_args = TrainingArguments(**base_kwargs)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collate_with_padding,
)
train_result = trainer.train()
print(f"loss: {getattr(train_result, 'training_loss', None)}")

# ─── Cell 8: PEFT 어댑터 저장 ─────────────────────────────────────────────────
model.save_pretrained(output_dir)
print('saved PEFT adapters:', output_dir)

# ─── Cell 9: 어댑터 재로드 ───────────────────────────────────────────────────
from peft import PeftModel
reload_kwargs = {'trust_remote_code': True}
if use_4bit:
    reload_kwargs.update({'load_in_4bit': True, 'device_map': 'auto'})
else:
    reload_kwargs.update({'torch_dtype': torch.bfloat16
                          if torch.cuda.is_available() else torch.float32})
base_model_for_eval = AutoModelForCausalLM.from_pretrained(model_name, **reload_kwargs)
model = PeftModel.from_pretrained(base_model_for_eval, output_dir)
if torch.cuda.is_available() and not use_4bit:
    model.to('cuda')
model.eval()
print('Eval model dtype:', next(model.parameters()).dtype)

# ─── Cell 10: 검증 평가 ───────────────────────────────────────────────────────
import random
label_list = list(LABEL_DESC.keys())

def predict_label_from_prompt(prompt_text, max_new_tokens=6):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated  = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    upper = generated.upper()
    for lb in label_list:
        if lb in upper: return lb, generated
    first = generated.split()[0].strip(""".,:;()[]{}"'""").upper() if generated else ''
    return (first if first in label_list else 'ETC'), generated

val_ds   = load_dataset('json', data_files={'validation': valid_file})['validation']
num_eval = min(100, len(val_ds))
indices  = random.sample(range(len(val_ds)), k=num_eval)
correct, samples = 0, []
for idx in indices:
    item = val_ds[idx]
    pred, raw = predict_label_from_prompt(item['prompt'])
    correct += int(pred == item['response'])
    if len(samples) < 5: samples.append((item['response'], pred, raw))

print(f'Validation accuracy ({num_eval}개): {correct/num_eval:.4f}')
for row in samples: print(row)

# ─── Cell 11: 단건 추론 테스트 ───────────────────────────────────────────────
test_clause = '신청인은 접수일로부터 14일 이내에 이의신청서를 제출해야 하며, 담당 기관은 30일 내에 심사 결과를 통지한다.'
label_candidates = ', '.join([f"{k}({v})" for k, v in LABEL_DESC.items()])
test_prompt = (f"다음 조항을 다음 라벨 중 하나로 분류하시오. "
               f"가능한 라벨: {label_candidates}\n조항: {test_clause}\n라벨:")
pred, raw = predict_label_from_prompt(test_prompt)
print('입력 조항:', test_clause)
print('예측 라벨:', pred, '/ 생성 원문:', raw)

CPU RAM: 3.6 GB 남음
GPU: NVIDIA GeForce RTX 5060 Ti
GPU VRAM: 17.1 GB 전체
GPU VRAM: 0.0 GB 사용 중
Config set: Qwen/Qwen2.5-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13178.92it/s]


model load
Selected target_modules for LoRA: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


Generating train split: 2000 examples [00:00, 181811.66 examples/s]
Generating validation split: 400 examples [00:00, 61385.30 examples/s]
Map: 100%|██████████| 400/400 [00:00<00:00, 3288.86 examples/s]


Datasets prepared: 2000 400


...

loss: 0.0069333093897524425
saved PEFT adapters: peft_output


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7333.72it/s]


Eval model dtype: torch.bfloat16


Generating validation split: 400 examples [00:00, 66666.20 examples/s]


Validation accuracy (100개): 1.0000
('ETC', 'ETC', 'ETC(시행일')
('CRIT', 'CRIT', 'CRIT(자격/요')
('ETC', 'ETC', 'ETC(시행일')
('ORG', 'ORG', 'ORG(ORG)')
('DEF', 'DEF', 'DEF(정의/목')
입력 조항: 신청인은 접수일로부터 14일 이내에 이의신청서를 제출해야 하며, 담당 기관은 30일 내에 심사 결과를 통지한다.
예측 라벨: PROC / 생성 원문: PROC(PROCEDURE)
